# Notebook 7: Phase-C Corruption Selective Correction

This notebook stress-tests `phase_c.pt` on corrupted CIFAR-10. It compares three readouts in each experiment cell:
- backbone alone
- backbone + full-`z` selective correction
- backbone + top-node subset selective correction

The top-node path is intentionally a **subset** of the strongest pair-specific nodes, not a single node.


In [ ]:
# Colab-first setup: clone/update the repo, install it, and mount Drive.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print("Running outside Colab; using the current working tree.")


In [ ]:
from flow_circuits.data import CIFAR10_CORRUPTION_NAMES
from flow_circuits.evaluation import (
    NB07_EXPERIMENT_IDS,
    run_corruption_sweep_experiment,
    run_top_node_subset_sweep_experiment,
)
from flow_circuits.training import load_components_from_checkpoint


## Config

Keep this notebook Phase-C-only. Adjust the corruption suite, trigger rule, and top-node subset settings below.


In [ ]:
from pathlib import Path

RUN_MODE = "full"  # "debug" or "full"
CONFIG_NAME = "resnet18_aligned"
EXPERIMENTS = "all"
FORCE_RERUN = False

TRIGGER_MODE = "hard_pair_top2_and_low_margin"
MARGIN_THRESHOLD = None
MARGIN_QUANTILE = 0.25
ENTROPY_THRESHOLD = None
ENTROPY_QUANTILE = 0.75

CORRUPTION_NAMES = ["gaussian_noise", "gaussian_blur", "contrast"]
SEVERITIES = [1, 3, 5]
TOP_NODE_FRACTION = 0.05
TOP_NODE_MIN_K = 3
TOP_NODE_MAX_K = 12

TOP_NODE_FRACTIONS = [0.02, 0.05, 0.10, 0.20]
SUBSET_SWEEP_CORRUPTION = "gaussian_noise"
SUBSET_SWEEP_SEVERITY = 3

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits")
EXPERIMENT_ROOT = DRIVE_ROOT / "experiments" / CONFIG_NAME
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb07_phase_c_corruption_selective_correction" / CONFIG_NAME

PHASE_C_CHECKPOINT = EXPERIMENT_ROOT / "phase_c.pt"


In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

CHECKPOINT_TAG = "phase_c"
CHECKPOINT_LABEL = "Phase C"
EXPERIMENT_LABELS = {
    "corruption_sweep": "Corruption Sweep",
    "top_node_subset_sweep": "Top-Node Subset Sweep",
}
AUTO_N_JOBS = max(1, min(8, (os.cpu_count() or 1) - 1 if (os.cpu_count() or 1) > 1 else 1))

if EXPERIMENTS == "all":
    SELECTED_EXPERIMENTS = list(NB07_EXPERIMENT_IDS)
else:
    SELECTED_EXPERIMENTS = [str(name) for name in EXPERIMENTS]
invalid = sorted(set(SELECTED_EXPERIMENTS) - set(NB07_EXPERIMENT_IDS))
if invalid:
    raise ValueError(f"Unknown experiments: {invalid}. Valid ids: {NB07_EXPERIMENT_IDS}")
invalid_corruptions = sorted(set(CORRUPTION_NAMES) - set(CIFAR10_CORRUPTION_NAMES))
if invalid_corruptions:
    raise ValueError(f"Unknown corruption names: {invalid_corruptions}. Valid names: {list(CIFAR10_CORRUPTION_NAMES)}")
if SUBSET_SWEEP_CORRUPTION not in CIFAR10_CORRUPTION_NAMES:
    raise ValueError(f"Unknown subset sweep corruption: {SUBSET_SWEEP_CORRUPTION}")
if not PHASE_C_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_C_CHECKPOINT}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = OUTPUT_DIR / CHECKPOINT_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS_BY_MODE = {
    "debug": {
        "corruption_sweep": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 1000, "top_pairs": 3, "n_jobs": AUTO_N_JOBS},
        "top_node_subset_sweep": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 1000, "top_pairs": 3, "n_jobs": AUTO_N_JOBS},
    },
    "full": {
        "corruption_sweep": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "top_pairs": 5, "n_jobs": AUTO_N_JOBS},
        "top_node_subset_sweep": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "top_pairs": 5, "n_jobs": AUTO_N_JOBS},
    },
}
if RUN_MODE not in SETTINGS_BY_MODE:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
components = load_components_from_checkpoint(PHASE_C_CHECKPOINT, device=device)
base_config = components.config

RESULTS = {}


def _should_run(experiment_id):
    return experiment_id in SELECTED_EXPERIMENTS


def _cache_path(experiment_id):
    return RESULTS_DIR / f"{experiment_id}.json"


def _format_seconds(seconds):
    seconds = max(int(seconds), 0)
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h {minutes}m {seconds}s"
    if minutes:
        return f"{minutes}m {seconds}s"
    return f"{seconds}s"


def _progress_logger(**event):
    stamp = time.strftime("%H:%M:%S")
    experiment = event.get("experiment")
    stage = str(event.get("stage", "")).replace("_", " ")
    completed = event.get("completed")
    total = event.get("total")
    elapsed = _format_seconds(event.get("elapsed_seconds", 0.0))
    eta = event.get("eta_seconds")
    message = event.get("message", "")
    if total:
        prefix = f"{stage} {completed}/{total}"
        eta_text = f" | ETA {_format_seconds(eta)}" if eta is not None else ""
    else:
        prefix = stage
        eta_text = ""
    print(f"[{stamp}] {CHECKPOINT_LABEL} | {experiment} | {prefix} | elapsed {elapsed}{eta_text} | {message}")


def _run_or_load(experiment_id, runner):
    cache_path = _cache_path(experiment_id)
    if cache_path.exists() and not FORCE_RERUN:
        print(f"Loading cached {experiment_id}: {cache_path}")
        return json.loads(cache_path.read_text(encoding="utf-8"))
    print(f"Running {experiment_id} ...")
    return runner(cache_path)


def _write_summary_table(rows, title):
    print(title)
    if not rows:
        print("(no rows)")
        return
    if pd is None:
        for row in rows:
            print(row)
        return
    display(pd.DataFrame(rows))


def _plot_model_triplet(rows, *, title, x_labels):
    if not rows:
        print(f"{title}: no rows")
        return
    x = np.arange(len(rows))
    backbone = [row["backbone_overall_accuracy"] for row in rows]
    full_z = [row["full_z_hybrid_overall_accuracy"] for row in rows]
    top_subset = [row["top_node_subset_overall_accuracy"] for row in rows]
    fig, ax = plt.subplots(figsize=(max(8, len(rows) * 0.8), 4.5))
    ax.plot(x, backbone, marker="o", label="Backbone")
    ax.plot(x, full_z, marker="o", label="Backbone + Full z")
    ax.plot(x, top_subset, marker="o", label="Backbone + Top-Node Subset")
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.set_ylabel("overall accuracy")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


## Experiment 1: Corruption Sweep

Run the same three-model comparison across a small corruption suite and severity ladder.


In [ ]:
if _should_run("corruption_sweep"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["corruption_sweep"]
    RESULTS["corruption_sweep"] = _run_or_load(
        "corruption_sweep",
        lambda cache_path: run_corruption_sweep_experiment(
            components,
            device=device,
            checkpoint_tag=CHECKPOINT_TAG,
            data_dir=base_config["data"]["data_dir"],
            batch_size=base_config["data"]["batch_size"],
            corruption_names=CORRUPTION_NAMES,
            severities=SEVERITIES,
            fit_max_images=settings["fit_max_images"],
            val_max_images=settings["val_max_images"],
            test_max_images=settings["test_max_images"],
            top_pairs=settings["top_pairs"],
            top_node_fraction=TOP_NODE_FRACTION,
            top_node_min_k=TOP_NODE_MIN_K,
            top_node_max_k=TOP_NODE_MAX_K,
            trigger_mode=TRIGGER_MODE,
            margin_threshold=MARGIN_THRESHOLD,
            margin_quantile=MARGIN_QUANTILE,
            entropy_threshold=ENTROPY_THRESHOLD,
            entropy_quantile=ENTROPY_QUANTILE,
            num_workers=base_config["data"].get("num_workers", 4),
            seed=base_config["data"].get("seed", 0),
            augment_fit=base_config["data"].get("augment_fit", True),
            download=base_config["data"].get("download", True),
            n_jobs=settings["n_jobs"],
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
    sweep = RESULTS["corruption_sweep"]
    _write_summary_table([sweep["summary"]], title="Corruption sweep summary")
    _write_summary_table(sweep["rows"], title="Corruption sweep rows")
    _plot_model_triplet(
        sweep["rows"],
        title="Corruption Sweep: Overall Accuracy",
        x_labels=[f"{row['corruption_name']}@{row['severity']}" for row in sweep["rows"]],
    )
else:
    print("Skipping corruption_sweep.")


## Experiment 2: Top-Node Subset Sweep

Sweep a **fractional subset** of the strongest pair-specific nodes for one representative corruption setting.


In [ ]:
if _should_run("top_node_subset_sweep"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["top_node_subset_sweep"]
    RESULTS["top_node_subset_sweep"] = _run_or_load(
        "top_node_subset_sweep",
        lambda cache_path: run_top_node_subset_sweep_experiment(
            components,
            device=device,
            checkpoint_tag=CHECKPOINT_TAG,
            data_dir=base_config["data"]["data_dir"],
            batch_size=base_config["data"]["batch_size"],
            corruption_name=SUBSET_SWEEP_CORRUPTION,
            severity=SUBSET_SWEEP_SEVERITY,
            top_node_fractions=TOP_NODE_FRACTIONS,
            fit_max_images=settings["fit_max_images"],
            val_max_images=settings["val_max_images"],
            test_max_images=settings["test_max_images"],
            top_pairs=settings["top_pairs"],
            top_node_min_k=TOP_NODE_MIN_K,
            top_node_max_k=TOP_NODE_MAX_K,
            trigger_mode=TRIGGER_MODE,
            margin_threshold=MARGIN_THRESHOLD,
            margin_quantile=MARGIN_QUANTILE,
            entropy_threshold=ENTROPY_THRESHOLD,
            entropy_quantile=ENTROPY_QUANTILE,
            num_workers=base_config["data"].get("num_workers", 4),
            seed=base_config["data"].get("seed", 0),
            augment_fit=base_config["data"].get("augment_fit", True),
            download=base_config["data"].get("download", True),
            n_jobs=settings["n_jobs"],
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
    subset = RESULTS["top_node_subset_sweep"]
    _write_summary_table([subset["summary"]], title="Top-node subset sweep summary")
    _write_summary_table(subset["rows"], title="Top-node subset sweep rows")
    _plot_model_triplet(
        subset["rows"],
        title=f"Top-Node Subset Sweep: {SUBSET_SWEEP_CORRUPTION}@{SUBSET_SWEEP_SEVERITY}",
        x_labels=[f"{row['top_node_fraction']:.2f} ({row['top_k_nodes']})" for row in subset["rows"]],
    )
else:
    print("Skipping top_node_subset_sweep.")


## Final Summary

This cell interprets the corruption results and tells you whether `z` looks broadly useful, narrowly useful, or mostly redundant under stress.


In [ ]:
summary_rows = []
interpretation_lines = []

if "corruption_sweep" in RESULTS:
    sweep = RESULTS["corruption_sweep"]
    rows = sweep["rows"]
    summary_rows.append({
        "experiment": "Corruption Sweep",
        "value": sweep["summary"]["mean_full_z_hybrid_overall_accuracy"],
        "metric": "mean_full_z_hybrid_overall_accuracy",
    })
    avg_full_gain = sweep["summary"]["mean_full_z_gain_vs_backbone"]
    avg_top_gain = sweep["summary"]["mean_top_node_subset_gain_vs_backbone"]
    best_full = max(rows, key=lambda row: row["full_z_gain_vs_backbone"])
    best_top = max(rows, key=lambda row: row["top_node_subset_gain_vs_backbone"])
    if avg_full_gain > 0.002:
        interpretation_lines.append(f"Full-z selective correction is meaningfully helpful under corruption on average (+{avg_full_gain:.4f} accuracy vs backbone).")
    elif avg_full_gain > 0:
        interpretation_lines.append(f"Full-z selective correction gives small but real average gains under corruption (+{avg_full_gain:.4f} accuracy vs backbone).")
    else:
        interpretation_lines.append(f"Full-z selective correction does not improve corrupted accuracy on average (gain {avg_full_gain:.4f}).")
    if avg_top_gain > 0:
        interpretation_lines.append(f"The top-node subset also helps on average, but usually less than full z (+{avg_top_gain:.4f} accuracy vs backbone).")
    else:
        interpretation_lines.append(f"The top-node subset does not improve corrupted accuracy on average (gain {avg_top_gain:.4f}).")
    interpretation_lines.append(
        f"The strongest full-z corruption win is {best_full['corruption_name']} severity {best_full['severity']} (gain {best_full['full_z_gain_vs_backbone']:.4f})."
    )
    interpretation_lines.append(
        f"The strongest top-node-subset corruption win is {best_top['corruption_name']} severity {best_top['severity']} (gain {best_top['top_node_subset_gain_vs_backbone']:.4f})."
    )

if "top_node_subset_sweep" in RESULTS:
    subset = RESULTS["top_node_subset_sweep"]
    rows = subset["rows"]
    summary_rows.append({
        "experiment": "Top-Node Subset Sweep",
        "value": subset["summary"]["best_top_node_subset_accuracy"],
        "metric": "best_top_node_subset_accuracy",
    })
    best = max(rows, key=lambda row: row["top_node_subset_overall_accuracy"])
    if best["top_node_subset_gap_to_full_z"] <= 0.001:
        interpretation_lines.append(
            f"A compact top-node subset is enough here: fraction {best['top_node_fraction']:.2f} ({best['top_k_nodes']} nodes) comes within {best['top_node_subset_gap_to_full_z']:.4f} of full-z accuracy."
        )
    elif best["top_node_subset_gap_to_full_z"] <= 0.003:
        interpretation_lines.append(
            f"A moderate top-node subset captures most of the useful signal: fraction {best['top_node_fraction']:.2f} ({best['top_k_nodes']} nodes) trails full z by only {best['top_node_subset_gap_to_full_z']:.4f}."
        )
    else:
        interpretation_lines.append(
            f"Useful correction signal still looks fairly distributed: even the best top-node subset (fraction {best['top_node_fraction']:.2f}, {best['top_k_nodes']} nodes) trails full z by {best['top_node_subset_gap_to_full_z']:.4f}."
        )

_write_summary_table(summary_rows, title="NB07 summary")
print("Interpretation")
for line in interpretation_lines:
    print(f"- {line}")
